# TILE OUTPUT AS ZARR

In [3]:
import os
import json
import re
import glob
import logging

import numpy as np
from natsort import natsorted
from tqdm.auto import tqdm
from skimage import io
import dask
import zarr

import napari
from macrohet import dataio, tile, visualise
# import btrack


scale_factor = 2 #5.04
ndim = 2

### Load experiment of choice

The Opera Phenix is a high-throughput confocal microscope that acquires very large 5-dimensional (TCZXY) images over several fields of view in any one experiment. Therefore, a lazy-loading approach is chosen to mosaic, view and annotate these images. This approach depends upon Dask and DaskFusion. The first step is to load the main metadata file (typically called `Index.idx.xml` and located in the main `Images` directory) that contains the image filenames and associated TCXZY information used to organise the images.

In [4]:

metadata_fn = f'/mnt/DATA3/BPP0050/BPP0050-1-Live-cell-to4i_Live-1__2025-04-09T18_25_04-Measurement 1/Images/Index.idx.xml'
metadata = dataio.read_harmony_metadata(metadata_fn)  

Reading metadata XML file...


0it [00:00, ?it/s]

Extracting metadata complete!


In [10]:
expt_ID = 'ND0000'
location = 'SYNO' # 'NEMO' # 'SYNO'
base_dir = f'/mnt/{location}/macrohet_{location.lower()}/data/{expt_ID}/'
metadata_fn = os.path.join(base_dir, 'acquisition/Images/Index.idx.xml')
metadata = dataio.read_harmony_metadata(metadata_fn)  

Reading metadata XML file...


0it [00:00, ?it/s]

Extracting metadata complete!


In [11]:
metadata

,id,State,URL,Row,Col,FieldID,PlaneID,TimepointID,ChannelID,FlimID,...,PositionZ,AbsPositionZ,MeasurementTimeOffset,AbsTime,MainExcitationWavelength,MainEmissionWavelength,ObjectiveMagnification,ObjectiveNA,ExposureTime,OrientationMatrix
0,0203K1F1P1R1,Ok,r02c03f01p01-ch1sk1fk1fl1.tiff,2,3,1,1,0,1,1,...,0,0.135205805,0,2023-08-04T15:28:16.5+01:00,561,599,40,1.1,0.2,"[[0.994928,0,0,15.1],[0,-0.994928,0,-5.3],[0,0..."
1,0203K1F1P1R2,Ok,r02c03f01p01-ch2sk1fk1fl1.tiff,2,3,1,1,0,2,1,...,0,0.135205805,0,2023-08-04T15:28:16.5+01:00,740,0,40,1.1,0.2,"[[0.994928,0,0,15.1],[0,-0.994928,0,-5.3],[0,0..."
2,0203K1F1P1R3,Ok,r02c03f01p01-ch3sk1fk1fl1.tiff,2,3,1,1,0,3,1,...,0,0.135205805,0,2023-08-04T15:28:16.767+01:00,640,706,40,1.1,0.2,"[[0.994928,0,0,15.1],[0,-0.994928,0,-5.3],[0,0..."
3,0203K1F1P2R1,Ok,r02c03f01p02-ch1sk1fk1fl1.tiff,2,3,1,2,0,1,1,...,2E-06,0.135207802,0,2023-08-04T15:28:17.047+01:00,561,599,40,1.1,0.2,"[[0.994928,0,0,15.1],[0,-0.994928,0,-5.3],[0,0..."
4,0203K1F1P2R2,Ok,r02c03f01p02-ch2sk1fk1fl1.tiff,2,3,1,2,0,2,1,...,2E-06,0.135207802,0,2023-08-04T15:28:17.047+01:00,740,0,40,1.1,0.2,"[[0.994928,0,0,15.1],[0,-0.994928,0,-5.3],[0,0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
243643,0310K376F9P2R2,Ok,r03c10f09p02-ch2sk376fk1fl1.tiff,3,10,9,2,375,2,1,...,2E-06,0.135064006,337503.973,2023-08-08T13:15:47.533+01:00,740,0,40,1.1,0.2,"[[0.994928,0,0,15.1],[0,-0.994928,0,-5.3],[0,0..."
243644,0310K376F9P2R3,Ok,r03c10f09p02-ch3sk376fk1fl1.tiff,3,10,9,2,375,3,1,...,2E-06,0.135064006,337503.973,2023-08-08T13:15:47.8+01:00,640,706,40,1.1,0.2,"[[0.994928,0,0,15.1],[0,-0.994928,0,-5.3],[0,0..."
243645,0310K376F9P3R1,Ok,r03c10f09p03-ch1sk376fk1fl1.tiff,3,10,9,3,375,1,1,...,4E-06,0.135066003,337503.973,2023-08-08T13:15:48.08+01:00,561,599,40,1.1,0.2,"[[0.994928,0,0,15.1],[0,-0.994928,0,-5.3],[0,0..."
243646,0310K376F9P3R2,Ok,r03c10f09p03-ch2sk376fk1fl1.tiff,3,10,9,3,375,2,1,...,4E-06,0.135066003,337503.973,2023-08-08T13:15:48.097+01:00,740,0,40,1.1,0.2,"[[0.994928,0,0,15.1],[0,-0.994928,0,-5.3],[0,0..."


## Alternative temporary method for suspected incomplete export

In [12]:
metadata[['Row', 'Col']].drop_duplicates()

,Row,Col
0,2,3
81,2,4
162,2,9
243,2,10
324,3,3
405,3,4
486,3,9
567,3,10


In [13]:
for i, (row, column) in metadata[['Row', 'Col']].drop_duplicates().iterrows():
    print(row, column)

2 3
2 4
2 9
2 10
3 3
3 4
3 9
3 10


### Now to lazily mosaic the images using Dask prior to saving them out as zarr

In [14]:
zarr_fns_to_do =[]
pattern = r'\((\d+), (\d+)\)\.zarr$'
# for acq_ID, data in tqdm(assay_layout.iterrows(), total = len(assay_layout)):
for i, (row, column) in metadata[['Row', 'Col']].drop_duplicates().iterrows():
    acq_ID = (int(row), int(column))
    zarr_fn = f'/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/{expt_ID}/acquisition/zarr/{acq_ID}.zarr'
    if not os.path.exists(zarr_fn):
        zarr_dir = (os.path.dirname(zarr_fn))
        os.makedirs(zarr_dir, exist_ok = True)
        zarr_fns_to_do.append(zarr_fn)

In [15]:
zarr_fns_to_do

['/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/ND0000/acquisition/zarr/(2, 3).zarr',
 '/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/ND0000/acquisition/zarr/(2, 4).zarr',
 '/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/ND0000/acquisition/zarr/(2, 9).zarr',
 '/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/ND0000/acquisition/zarr/(2, 10).zarr',
 '/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/ND0000/acquisition/zarr/(3, 3).zarr',
 '/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/ND0000/acquisition/zarr/(3, 4).zarr',
 '/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/ND0000/acquisition/zarr/(3, 9).zarr',
 '/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/N

In [16]:
len(zarr_fns_to_do)

8

In [17]:
zarr_fn

'/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/ND0000/acquisition/zarr/(3, 10).zarr'

In [18]:
re.search(pattern, zarr_fn)

<re.Match object; span=(115, 127), match='(3, 10).zarr'>

In [23]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger()

for zarr_fn in tqdm(zarr_fns_to_do, total=len(zarr_fns_to_do), desc='Iterating over zarr fns to do'):
    match = re.search(pattern, zarr_fn)
    row, column = acq_ID = (int(match.group(1)), int(match.group(2)))

    image_dir = os.path.join(base_dir, 'acquisition/Images')
    try:
        logger.info(f"Processing acquisition ID: {acq_ID}")
        
        logger.info(f"Compiling mosaic for row {row}, column {column} from {image_dir}")
        dask_images = tile.compile_mosaic(image_dir, metadata, row, column).compute()
        logger.info(f"Mosaic compiled successfully for acquisition ID: {acq_ID}")
        
        # data = dict() #assay_layout.loc[acq_ID]
        logger.info(f"Retrieved assay layout data for acquisition ID: {acq_ID}")
        
        zarr_group = zarr.open(zarr_fn, mode='w')
        logger.info(f"Opened Zarr file: {zarr_fn}")
        
        acq_metadata = dict()
        acq_metadata['Acquisition ID'] = acq_ID
        acq_metadata['Experiment ID'] = expt_ID
        acq_metadata['Dimensionality'] = 'TCZYX'
        
        logger.info(f"Rechunking images for acquisition ID: {acq_ID}")
        rechunked_images = dask_images.rechunk((1, 1, 1, 6048, 6048))
        
        logger.info(f"Loading image into Zarr for acquisition ID: {acq_ID}")
        dask.array.to_zarr(rechunked_images, zarr_fn, component='images')
        logger.info(f"Image loaded into Zarr for acquisition ID: {acq_ID}")
        
        zarr_group.attrs['metadata'] = acq_metadata
        logger.info(f"Metadata saved for acquisition ID: {acq_ID}")
        
        logger.info(f"Successfully processed acquisition ID: {acq_ID}")
    except Exception as e:
        logger.error(f"An error occurred: {e}")
        logger.error(f"Acquisition ID: {acq_ID}, Error: {e}")  
        continue

Iterating over zarr fns to do:   0%|          | 0/8 [00:00<?, ?it/s]

INFO:root:Processing acquisition ID: (2, 3)
INFO:root:Compiling mosaic for row 2, column 3 from /mnt/SYNO/macrohet_syno/data/ND0000/acquisition/Images
INFO:root:Mosaic compiled successfully for acquisition ID: (2, 3)
INFO:root:Retrieved assay layout data for acquisition ID: (2, 3)
INFO:root:Opened Zarr file: /run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/ND0000/acquisition/zarr/(2, 3).zarr
INFO:root:Rechunking images for acquisition ID: (2, 3)
INFO:root:Loading image into Zarr for acquisition ID: (2, 3)
ERROR:root:An error occurred: 'images/8.2.0.0.0'
ERROR:root:Acquisition ID: (2, 3), Error: 'images/8.2.0.0.0'
INFO:root:Processing acquisition ID: (2, 4)
INFO:root:Compiling mosaic for row 2, column 4 from /mnt/SYNO/macrohet_syno/data/ND0000/acquisition/Images
INFO:root:Mosaic compiled successfully for acquisition ID: (2, 4)
INFO:root:Retrieved assay layout data for acquisition ID: (2, 4)
ERROR:root:An error occurred: '.zgroup'
ERROR:root:Acqu

# Doing Victor's first

### Move files to acquisition subdir for consistency

In [ ]:
src_dir = '/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,volume=OPERA/Nathan_temp/20250312 pTimer Sautons Live BODIPY__2025-03-12T16_53_21-Measurement 1'

In [63]:
expt_ID = 'VP0000'
location = 'SYNO' # 'NEMO' # 'SYNO'
# base_dir = f'/mnt/{location}/macrohet_{location.lower()}/data/{expt_ID}/'
base_dir = f'/mnt/OPERA/Nathan_temp/{expt_ID}'
metadata_fn = os.path.join(base_dir, 'acquisition/Images/Index.idx.xml')
metadata = dataio.read_harmony_metadata(metadata_fn)  

Reading metadata XML file...


0it [00:00, ?it/s]

Extracting metadata complete!


In [69]:
metadata.dropna()

,id,State,URL,Row,Col,FieldID,PlaneID,TimepointID,ChannelID,FlimID,...,PositionZ,AbsPositionZ,MeasurementTimeOffset,AbsTime,MainExcitationWavelength,MainEmissionWavelength,ObjectiveMagnification,ObjectiveNA,ExposureTime,OrientationMatrix
0,0102K1F1P1R1,Ok,r01c02f01p01-ch1sk1fk1fl1.tiff,1,2,1,1,0,1,1,...,3.559E-06,0.135277495,0,2025-03-12T16:53:44.16+00:00,740,0,40,1.1,0.1,"[[0.988915,0,0,-12.4],[0,-0.988915,0,-14.4],[0..."
1,0102K1F1P1R2,Ok,r01c02f01p01-ch2sk1fk1fl1.tiff,1,2,1,1,0,2,1,...,0,0.135283396,0,2025-03-12T16:53:44.66+00:00,488,522,40,1.1,0.16,"[[0.988915,0,0,-12.4],[0,-0.988915,0,-14.4],[0..."
2,0102K1F1P1R3,Ok,r01c02f01p01-ch3sk1fk1fl1.tiff,1,2,1,1,0,3,1,...,0,0.135283396,0,2025-03-12T16:53:45.067+00:00,561,599,40,1.1,0.2,"[[0.988915,0,0,-12.4],[0,-0.988915,0,-14.4],[0..."
3,0102K1F1P1R4,Ok,r01c02f01p01-ch4sk1fk1fl1.tiff,1,2,1,1,0,4,1,...,0,0.135283396,0,2025-03-12T16:53:45.08+00:00,740,0,40,1.1,0.1,"[[0.988915,0,0,-12.4],[0,-0.988915,0,-14.4],[0..."
4,0102K1F1P2R1,Ok,r01c02f01p02-ch1sk1fk1fl1.tiff,1,2,1,2,0,1,1,...,3.559E-06,0.135277495,0,2025-03-12T16:53:44.16+00:00,740,0,40,1.1,0.1,"[[0.988915,0,0,-12.4],[0,-0.988915,0,-14.4],[0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1249835,0811K50F25P4R4,Ok,r08c11f25p04-ch4sk50fk1fl1.tiff,8,11,25,4,49,4,1,...,3E-06,0.135085896,253572.12699999998,2025-03-15T16:33:57.833+00:00,740,0,40,1.1,0.1,"[[0.988915,0,0,-12.4],[0,-0.988915,0,-14.4],[0..."
1249836,0811K50F25P5R1,Ok,r08c11f25p05-ch1sk50fk1fl1.tiff,8,11,25,5,49,1,1,...,3.559E-06,0.135077,253572.12699999998,2025-03-15T16:33:55.073+00:00,740,0,40,1.1,0.1,"[[0.988915,0,0,-12.4],[0,-0.988915,0,-14.4],[0..."
1249837,0811K50F25P5R2,Ok,r08c11f25p05-ch2sk50fk1fl1.tiff,8,11,25,5,49,2,1,...,4E-06,0.135086894,253572.12699999998,2025-03-15T16:33:58.13+00:00,488,522,40,1.1,0.16,"[[0.988915,0,0,-12.4],[0,-0.988915,0,-14.4],[0..."
1249838,0811K50F25P5R3,Ok,r08c11f25p05-ch3sk50fk1fl1.tiff,8,11,25,5,49,3,1,...,4E-06,0.135086894,253572.12699999998,2025-03-15T16:33:58.457+00:00,561,599,40,1.1,0.2,"[[0.988915,0,0,-12.4],[0,-0.988915,0,-14.4],[0..."


In [70]:
from tqdm.auto import tqdm

In [65]:
metadata_path = glob.glob(os.path.join(base_dir, 'acquisition/Assaylayout/*.xml'))[0]
assay_layout = dataio.read_harmony_metadata(metadata_path, assay_layout=True,replicate_number=False)# mask_exist=True,  image_dir = image_dir, image_metadata = metadata)
assay_layout

Reading metadata XML file...
Extracting metadata complete!


Compound Bacteria
Row Column                                 
1   2               INH 0.04 ug/ml      A15
    4               INH 0.04 ug/ml      A60
    6               RIF 0.01 ug/ml      A15
    8               RIF 0.01 ug/ml      A60
    10                EMB 0.2ug/ml      A15
    11                EMB 0.2ug/ml      A60
2   2               INH 0.08 ug/ml      A15
    4               INH 0.08 ug/ml      A60
    6               RIF 0.02 ug/ml      A15
    8               RIF 0.02 ug/ml      A60
    10                EMB 0.4ug/ml      A15
    11                EMB 0.4ug/ml      A60
3   2               INH 0.16 ug/ml      A15
    4               INH 0.16 ug/ml      A60
    6               RIF 0.04 ug/ml      A15
    8               RIF 0.04 ug/ml      A60
    10                EMB 0.8ug/ml      A15
    11                EMB 0.8ug/ml      A60
4   2               INH 0.32 ug/ml      A15
    4               INH 0.32 ug/ml      A60
    6               RIF 0.08 ug/ml      A15
    8               RIF 0.08 ug/ml      A60
    10               EMB 1.16ug/ml      A15
    11               EMB 1.16ug/ml      A60
5   2               INH 0.64 ug/ml      A15
    4               INH 0.64 ug/ml      A60
    6               RIF 0.16 ug/ml      A15
    8               RIF 0.16 ug/ml      A60
    10               HEMB 3.2ug/ml      A15
    11                EMB 3.2ug/ml      A60
6   2                 PZA 025ug/ml      A15
    4                 PZA 025ug/ml      A60
    6                 PZA 200ug/ml      A15
    8                 PZA 200ug/ml      A60
    9              Bodipy Bacteria      A15
    11             Bodipy Bacteria      A60
7   2                 PZA 050ug/ml      A15
    4                 PZA 050ug/ml      A60
    6                 PZA 400ug/ml      A15
    8                 PZA 400ug/ml      A60
    9                  Bodipy Cell      A15
    11                 Bodipy Cell      A60
8   2                 PZA 100ug/ml      A15
    4                 PZA 100ug/ml      A60
    5                   pTimer 7H9      7H9
    6              Bodipy Bacteria      7H9
    7                  Bodipy Cell      7H9
    8       Bodipy Bacteria + Cell      7H9
    9       Bodipy Bacteria + Cell      A15
    11      Bodipy Bacteria + Cell      A60
1   1                          NaN      A15

In [66]:
zarr_fns_to_do =[]
pattern = r'\((\d+), (\d+)\)\.zarr$'
# for acq_ID, data in tqdm(assay_layout.iterrows(), total = len(assay_layout)):
for i, (row, column) in metadata[['Row', 'Col']].drop_duplicates().iterrows():
    acq_ID = (int(row), int(column))
    zarr_fn = f'/mnt/OPERA/Nathan_temp/{expt_ID}/acquisition/zarr/{acq_ID}.zarr'
    if not os.path.exists(zarr_fn):
        zarr_dir = (os.path.dirname(zarr_fn))
        os.makedirs(zarr_dir, exist_ok = True)
        zarr_fns_to_do.append(zarr_fn)

In [67]:
zarr_fns_to_do

['/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(1, 2).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(1, 4).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(1, 6).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(1, 8).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(1, 10).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(1, 11).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(2, 2).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(2, 4).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(2, 6).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(2, 8).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(2, 10).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(2, 11).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(3, 2).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(3, 4).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(3, 6).zarr',
 '/mnt/OPERA/Nathan_temp/VP0000/acquisition/zarr/(3

# DPC

In [74]:
expt_dir = base_dir

In [80]:
import os
import json
import re
import glob
import logging

import numpy as np
from natsort import natsorted
from tqdm.auto import tqdm
from skimage import io
import dask
import zarr

import napari
from macrohet import dataio, tile, visualise
import btrack

import os
import numpy as np
import zarr
import tifffile as tiff
import pandas as pd
from tqdm.auto import tqdm
from skimage.io import imread

In [ ]:
expt = expt_ID
# Load and process metadata
# metadata_fn = os.path.join(expt_dir, 'acquisition/Images/Index.idx.xml')
# try:
#     metadata = dataio.read_harmony_metadata(metadata_fn, iter=False)
# except Exception as e:
#     print(f"Error loading metadata for {expt}: {e}")
#     # continue

# Convert applicable columns to numeric values
metadata = metadata.apply(pd.to_numeric, errors='ignore')
# Invert Y coordinates (cathode ray tube adjustment)
metadata['PositionY'] = -metadata['PositionY']


# Find the minimum Z position for non-Phase Contrast channels
min_z = metadata.loc[~metadata['ChannelName'].str.contains('Phase Contrast', case=False, na=False), 'PositionZ'].min()

# Assign this minimum Z value to all "Phase Contrast" channels
metadata.loc[metadata['ChannelName'].str.contains('Phase Contrast', case=False, na=False), 'PositionZ'] = min_z


# Set image directory
image_directory = os.path.join(expt_dir, 'acquisition/Images')

# Create the output directory for TIFF files
output_dir = os.path.join(expt_dir, 'acquisition/tiff_images')
os.makedirs(output_dir, exist_ok=True)


for index, (row, column) in tqdm(metadata[['Row', 'Col']].drop_duplicates().iterrows(), 
                                 # desc=f"Processing Row {row}, Col {column}", 
                                 leave=False, 
                                 total=len(metadata[['Row', 'Col']].drop_duplicates())):

    acq_ID = row, column
    # Define output paths
    output_fn = f'row{row}_col{column}_{expt}.tiff'
    output_tiff_path = os.path.join(output_dir, output_fn)
    
    # Zarr output path
    zarr_output_dir = os.path.join(expt_dir, f'acquisition/zarr/({row}, {column}).zarr')
    os.makedirs(zarr_output_dir, exist_ok=True)
    
    # Filter metadata for the current row and column (include all time points)
    position_metadata = metadata[(metadata['Row'] == row) & (metadata['Col'] == column)]
    
    # Get unique time points
    time_points = sorted(position_metadata['TimepointID'].unique())
    
    # Initialize a list to store time-lapse volumes
    time_volumes = []
    
    # Iterate over each time point
    for t_point in tqdm(time_points, desc="Processing Time Points", leave=False):
        
        # Filter metadata for the current time point
        time_metadata = position_metadata[position_metadata['TimepointID'] == t_point]
    
        # Get unique Channel and Z positions
        channel_ids = sorted(time_metadata['ChannelID'].unique())
        z_positions = sorted(time_metadata['PositionZ'].unique())
    
        # Initialize a list to hold each channel's Z-stack for this time point
        channel_volumes = []
    
        # Iterate over each channel
        for channel_id in tqdm(channel_ids, desc=f"Processing Channels (T={t_point})", leave=False):
    
            channel_slice_metadata = time_metadata[time_metadata['ChannelID'] == channel_id]
            print(channel_slice_metadata['ChannelName'].iloc[0])
    
            z_slices = []
            
            # Iterate over Z positions
            for z_position in tqdm(z_positions, desc="Processing Z-slices", leave=False):
                z_slice_metadata = channel_slice_metadata[channel_slice_metadata['PositionZ'] == z_position]
    
                mosaic_slice = None
    
                # Process each image in the Z slice
                for _, row_ in tqdm(z_slice_metadata.iterrows(), desc="Placing Images", leave=False, total=len(z_slice_metadata)):
                    img_path = os.path.join(image_directory, row_['URL'])
                    try:
                        img = imread(img_path)  # Load the image
                    except FileNotFoundError:
                        img = np.zeros((row_['ImageSizeY'], row_['ImageSizeX']), dtype=np.uint16)
    
                    # Convert PositionX and PositionY to pixel coordinates
                    try:
                        x_pixel = int((row_['PositionX'] - z_slice_metadata['PositionX'].min()) / row_['ImageResolutionX'])
                        y_pixel = int((row_['PositionY'] - z_slice_metadata['PositionY'].min()) / row_['ImageResolutionY'])
                    except ZeroDivisionError:
                        print(f"Error in calculating pixel coordinates for {img_path}")
                        continue
    
                    # Initialize mosaic_slice if needed
                    if mosaic_slice is None:
                        mosaic_size_x = int((z_slice_metadata['PositionX'].max() - z_slice_metadata['PositionX'].min()) / row_['ImageResolutionX']) + row_['ImageSizeX']
                        mosaic_size_y = int((z_slice_metadata['PositionY'].max() - z_slice_metadata['PositionY'].min()) / row_['ImageResolutionY']) + row_['ImageSizeY']
                        mosaic_slice = np.zeros((mosaic_size_y, mosaic_size_x), dtype=np.uint16)
    
                    # Place the image in the mosaic
                    try:
                        existing_slice = mosaic_slice[y_pixel:y_pixel+img.shape[0], x_pixel:x_pixel+img.shape[1]]
                        np.maximum(existing_slice, img, out=existing_slice)
                    except ValueError:
                        print(f"Error placing image {row_['URL']} at position ({x_pixel}, {y_pixel}) in the mosaic.")
                        continue
    
                if mosaic_slice is None:
                    mosaic_slice = np.zeros((mosaic_size_y, mosaic_size_x), dtype=np.uint16)  
    
                if "Phase Contrast" in channel_slice_metadata['ChannelName'].iloc[0]:  
                    print(f"Distributing DPC image across {len(z_positions)} Z slices")
                    dpc_slice = mosaic_slice / len(z_positions)  
                    z_slices = [dpc_slice] * len(z_positions)  
                    break  
                else:
                    z_slices.append(mosaic_slice)
    
            if len(z_slices) > 0:
                try:
                    channel_volume = np.stack(z_slices, axis=0)
                    channel_volumes.append(channel_volume)
                except ValueError as e:
                    print(f"Error stacking Z slices for Channel {channel_id} at (T={t_point}, Row {row}, Col {column}): {e}")
    
        if len(channel_volumes) > 0:
            try:
                image_volume_czyx = np.stack(channel_volumes, axis=0)  # C, Z, Y, X
                time_volumes.append(image_volume_czyx)  # Store for this time point
            except ValueError as e:
                print(f"Error stacking channel volumes for T={t_point}, Row {row}, Col {column}: {e}")
    
    # Stack all time points into a 5D array (T, C, Z, Y, X)
    if len(time_volumes) > 0:
        try:
            image_volume_final = np.stack(time_volumes, axis=0)  # T, C, Z, Y, X
            print(f"Final image volume shape: {image_volume_final.shape}")
            # Rearrange the axes to FIJI format (T, Z, C, Y, X)
            image_volume_fiji_format = np.moveaxis(image_volume_final, 2, 1)  # Move Z (axis=2) to position 1
    
            # Save as TIFF
            # Save as TIFF
            tiff.imwrite(output_tiff_path, 
                         image_volume_fiji_format.astype(np.uint16), 
                         imagej=True, 
                         resolution=(1/(1E6*metadata['ImageResolutionX'].iloc[0]), 1/(1E6*metadata['ImageResolutionY'].iloc[0])), 
                         metadata={'unit': 'um', 'axes': 'TZCYX'})
            
            # Save as Zarr
            dask_images = dask.array.from_array(image_volume_final)
            zarr_group = zarr.open(zarr_output_dir, mode='w')
            rechunked_images = dask_images.rechunk((1, 1, 1, img.shape[0], img.shape[1]))
            dask.array.to_zarr(rechunked_images, zarr_output_dir, component='images')
            print(f"Saved Zarr array for (Row {row}, Col {column}) at {zarr_output_dir}")
    
        except Exception as e:
            print(f"Error saving files for Row {row}, Col {column}: {e}")

/tmp/ipykernel_37067/2942038205.py:11: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  metadata = metadata.apply(pd.to_numeric, errors='ignore')


  0%|          | 0/50 [00:00<?, ?it/s]

Processing Time Points:   0%|          | 0/50 [00:00<?, ?it/s]

Processing Channels (T=0):   0%|          | 0/4 [00:00<?, ?it/s]

Digital Phase Contrast_BPP0042


...

Placing Images:   0%|          | 0/25 [00:00<?, ?it/s]

Placing Images:   0%|          | 0/25 [00:00<?, ?it/s]

Placing Images:   0%|          | 0/25 [00:00<?, ?it/s]

Placing Images:   0%|          | 0/25 [00:00<?, ?it/s]

Placing Images:   0%|          | 0/25 [00:00<?, ?it/s]

In [68]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger()

for zarr_fn in tqdm(zarr_fns_to_do, total=len(zarr_fns_to_do), desc='Iterating over zarr fns to do'):
    match = re.search(pattern, zarr_fn)
    row, column = acq_ID = (int(match.group(1)), int(match.group(2)))

    image_dir = os.path.join(base_dir, 'acquisition/Images')
    try:
        logger.info(f"Processing acquisition ID: {acq_ID}")
        
        logger.info(f"Compiling mosaic for row {row}, column {column} from {image_dir}")
        dask_images = tile.compile_mosaic(image_dir, metadata, row, column).compute()
        logger.info(f"Mosaic compiled successfully for acquisition ID: {acq_ID}")
        
        data = assay_layout.loc[acq_ID]
        logger.info(f"Retrieved assay layout data for acquisition ID: {acq_ID}")
        
        zarr_group = zarr.open(zarr_fn, mode='w')
        logger.info(f"Opened Zarr file: {zarr_fn}")
        
        acq_metadata = data.to_dict()
        acq_metadata['Acquisition ID'] = acq_ID
        acq_metadata['Experiment ID'] = expt_ID
        acq_metadata['Dimensionality'] = 'TCZYX'
        
        logger.info(f"Rechunking images for acquisition ID: {acq_ID}")
        rechunked_images = dask_images.rechunk((1, 1, 1, 1080, 1080))
        
        logger.info(f"Loading image into Zarr for acquisition ID: {acq_ID}")
        dask.array.to_zarr(rechunked_images, zarr_fn, component='images')
        logger.info(f"Image loaded into Zarr for acquisition ID: {acq_ID}")
        
        zarr_group.attrs['metadata'] = acq_metadata
        logger.info(f"Metadata saved for acquisition ID: {acq_ID}")
        
        logger.info(f"Successfully processed acquisition ID: {acq_ID}")
    except Exception as e:
        logger.error(f"An error occurred: {e}")
        logger.error(f"Acquisition ID: {acq_ID}, Error: {e}")  
        continue

Iterating over zarr fns to do:   0%|          | 0/50 [00:00<?, ?it/s]

INFO:root:Processing acquisition ID: (1, 2)
INFO:root:Compiling mosaic for row 1, column 2 from /mnt/OPERA/Nathan_temp/VP0000/acquisition/Images
ERROR:root:An error occurred: [Errno 2] No such file or directory: '/mnt/OPERA/Nathan_temp/VP0000/acquisition/Images/r01c02f01p01-ch1sk1fk1fl1.tiff'
ERROR:root:Acquisition ID: (1, 2), Error: [Errno 2] No such file or directory: '/mnt/OPERA/Nathan_temp/VP0000/acquisition/Images/r01c02f01p01-ch1sk1fk1fl1.tiff'
INFO:root:Processing acquisition ID: (1, 4)
INFO:root:Compiling mosaic for row 1, column 4 from /mnt/OPERA/Nathan_temp/VP0000/acquisition/Images
ERROR:root:An error occurred: [Errno 2] No such file or directory: '/mnt/OPERA/Nathan_temp/VP0000/acquisition/Images/r01c04f01p01-ch1sk1fk1fl1.tiff'
ERROR:root:Acquisition ID: (1, 4), Error: [Errno 2] No such file or directory: '/mnt/OPERA/Nathan_temp/VP0000/acquisition/Images/r01c04f01p01-ch1sk1fk1fl1.tiff'
INFO:root:Processing acquisition ID: (1, 6)
INFO:root:Compiling mosaic for row 1, column 6

KeyboardInterrupt: 

# Now do ND1

In [24]:
expt_ID = 'ND0001'
location = 'SYNO' # 'NEMO' # 'SYNO'
base_dir = f'/mnt/{location}/macrohet_{location.lower()}/data/{expt_ID}/'
metadata_fn = os.path.join(base_dir, 'acquisition/Images/Index.idx.xml')
metadata = dataio.read_harmony_metadata(metadata_fn)  

Reading metadata XML file...


0it [00:00, ?it/s]

Extracting metadata complete!


In [25]:
metadata

,id,State,URL,Row,Col,FieldID,PlaneID,TimepointID,ChannelID,FlimID,...,PositionZ,AbsPositionZ,MeasurementTimeOffset,AbsTime,MainExcitationWavelength,MainEmissionWavelength,ObjectiveMagnification,ObjectiveNA,ExposureTime,OrientationMatrix
0,0301K1F1P1R1,Ok,r03c01f01p01-ch1sk1fk1fl1.tiff,3,1,1,1,0,1,1,...,-2E-06,0.135358006,0,2023-10-27T14:13:10.393+01:00,561,599,40,1.1,0.2,"[[0.995374,0,0,12.4],[0,-0.995374,0,-6.9],[0,0..."
1,0301K1F1P1R2,Ok,r03c01f01p01-ch2sk1fk1fl1.tiff,3,1,1,1,0,2,1,...,-2E-06,0.135358006,0,2023-10-27T14:13:10.393+01:00,740,0,40,1.1,0.2,"[[0.995374,0,0,12.4],[0,-0.995374,0,-6.9],[0,0..."
2,0301K1F1P1R3,Ok,r03c01f01p01-ch3sk1fk1fl1.tiff,3,1,1,1,0,3,1,...,-2E-06,0.135358006,0,2023-10-27T14:13:10.66+01:00,640,706,40,1.1,0.2,"[[0.995374,0,0,12.4],[0,-0.995374,0,-6.9],[0,0..."
3,0301K1F1P2R1,Ok,r03c01f01p02-ch1sk1fk1fl1.tiff,3,1,1,2,0,1,1,...,0,0.135360003,0,2023-10-27T14:13:10.94+01:00,561,599,40,1.1,0.2,"[[0.995374,0,0,12.4],[0,-0.995374,0,-6.9],[0,0..."
4,0301K1F1P2R2,Ok,r03c01f01p02-ch2sk1fk1fl1.tiff,3,1,1,2,0,2,1,...,0,0.135360003,0,2023-10-27T14:13:10.94+01:00,740,0,40,1.1,0.2,"[[0.995374,0,0,12.4],[0,-0.995374,0,-6.9],[0,0..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020595,0612K300F9P2R2,Ok,r06c12f09p02-ch2sk300fk1fl1.tiff,6,12,9,2,299,2,1,...,0,0.135085002,266674.747,2023-10-30T15:32:33.063+00:00,740,0,40,1.1,0.2,"[[0.995374,0,0,12.4],[0,-0.995374,0,-6.9],[0,0..."
1020596,0612K300F9P2R3,Ok,r06c12f09p02-ch3sk300fk1fl1.tiff,6,12,9,2,299,3,1,...,0,0.135085002,266674.747,2023-10-30T15:32:33.327+00:00,640,706,40,1.1,0.2,"[[0.995374,0,0,12.4],[0,-0.995374,0,-6.9],[0,0..."
1020597,0612K300F9P3R1,Ok,r06c12f09p03-ch1sk300fk1fl1.tiff,6,12,9,3,299,1,1,...,2E-06,0.135086998,266674.747,2023-10-30T15:32:33.61+00:00,561,599,40,1.1,0.2,"[[0.995374,0,0,12.4],[0,-0.995374,0,-6.9],[0,0..."
1020598,0612K300F9P3R2,Ok,r06c12f09p03-ch2sk300fk1fl1.tiff,6,12,9,3,299,2,1,...,2E-06,0.135086998,266674.747,2023-10-30T15:32:33.627+00:00,740,0,40,1.1,0.2,"[[0.995374,0,0,12.4],[0,-0.995374,0,-6.9],[0,0..."


In [40]:
zarr_fns_to_do =[]
pattern = r'\((\d+), (\d+)\)\.zarr$'
# for acq_ID, data in tqdm(assay_layout.iterrows(), total = len(assay_layout)):
for i, (row, column) in metadata[['Row', 'Col']].drop_duplicates().iterrows():
    acq_ID = (int(row), int(column))
    zarr_fn = f'/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/{expt_ID}/acquisition/zarr/{acq_ID}.zarr'
    if not os.path.exists(zarr_fn):
        zarr_dir = (os.path.dirname(zarr_fn))
        os.makedirs(zarr_dir, exist_ok = True)
        zarr_fns_to_do.append(zarr_fn)

## Alternative temporary method for suspected incomplete export

In [26]:
metadata[['Row', 'Col']].drop_duplicates()

,Row,Col
0,3,1
81,3,2
162,3,3
243,3,4
324,3,5
405,3,6
486,3,7
567,3,8
648,3,9
729,3,10


In [27]:
for i, (row, column) in metadata[['Row', 'Col']].drop_duplicates().iterrows():
    print(row, column)

3 1
3 2
3 3
3 4
3 5
3 6
3 7
3 8
3 9
3 10
3 11
3 12
4 3
4 4
4 5
4 6
4 7
4 8
4 9
4 10
4 11
4 12
5 3
5 4
5 5
5 6
5 7
5 8
5 9
5 10
5 11
5 12
6 3
6 4
6 5
6 6
6 7
6 8
6 9
6 10
6 11
6 12


### Now to lazily mosaic the images using Dask prior to saving them out as zarr

In [31]:
os.path.exists(f'/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/')

False

In [32]:
zarr_fns_to_do =[]
pattern = r'\((\d+), (\d+)\)\.zarr$'
# for acq_ID, data in tqdm(assay_layout.iterrows(), total = len(assay_layout)):
for i, (row, column) in metadata[['Row', 'Col']].drop_duplicates().iterrows():
    acq_ID = (int(row), int(column))
    zarr_fn = f'/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA/Nathan_temp/{expt_ID}/acquisition/zarr/{acq_ID}.zarr'
    if not os.path.exists(zarr_fn):
        zarr_dir = (os.path.dirname(zarr_fn))
        os.makedirs(zarr_dir, exist_ok = True)
        zarr_fns_to_do.append(zarr_fn)

OSError: [Errno 5] Input/output error: '/run/user/30046150/gvfs/afp-volume:host=DS3617xs.local,user=ADMIN,volume=OPERA'

In [ ]:
zarr_fns_to_do

In [ ]:
len(zarr_fns_to_do)

In [ ]:
zarr_fn

In [ ]:
re.search(pattern, zarr_fn)

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger()

for zarr_fn in tqdm(zarr_fns_to_do, total=len(zarr_fns_to_do), desc='Iterating over zarr fns to do'):
    match = re.search(pattern, zarr_fn)
    row, column = acq_ID = (int(match.group(1)), int(match.group(2)))

    image_dir = os.path.join(base_dir, 'acquisition/Images')
    try:
        logger.info(f"Processing acquisition ID: {acq_ID}")
        
        logger.info(f"Compiling mosaic for row {row}, column {column} from {image_dir}")
        dask_images = tile.compile_mosaic(image_dir, metadata, row, column).compute()
        logger.info(f"Mosaic compiled successfully for acquisition ID: {acq_ID}")
        
        # data = dict() #assay_layout.loc[acq_ID]
        logger.info(f"Retrieved assay layout data for acquisition ID: {acq_ID}")
        
        zarr_group = zarr.open(zarr_fn, mode='w')
        logger.info(f"Opened Zarr file: {zarr_fn}")
        
        acq_metadata = dict()
        acq_metadata['Acquisition ID'] = acq_ID
        acq_metadata['Experiment ID'] = expt_ID
        acq_metadata['Dimensionality'] = 'TCZYX'
        
        logger.info(f"Rechunking images for acquisition ID: {acq_ID}")
        rechunked_images = dask_images.rechunk((1, 1, 1, 6048, 6048))
        
        logger.info(f"Loading image into Zarr for acquisition ID: {acq_ID}")
        dask.array.to_zarr(rechunked_images, zarr_fn, component='images')
        logger.info(f"Image loaded into Zarr for acquisition ID: {acq_ID}")
        
        zarr_group.attrs['metadata'] = acq_metadata
        logger.info(f"Metadata saved for acquisition ID: {acq_ID}")
        
        logger.info(f"Successfully processed acquisition ID: {acq_ID}")
    except Exception as e:
        logger.error(f"An error occurred: {e}")
        logger.error(f"Acquisition ID: {acq_ID}, Error: {e}")
        continue

## Quick check that it worked

In [ ]:
assay_layout

In [10]:
acq_ID = (5, 3)
zarr_fn = f'/mnt/SYNO/macrohet_syno/data/{expt_ID}/acquisition/zarr/{acq_ID}.zarr'

In [11]:
zarr_group = zarr.open(zarr_fn)

In [12]:
zarr_group.attrs['metadata']

{'Acquisition ID': [5, 5],
 'Compound': 'PZA',
 'Concentration': '60',
 'ConcentrationEC': 'EC50',
 'Dimensionality': 'TCZYX',
 'Experiment ID': 'ND0003',
 'Replicate #': 1,
 'Strain': 'RD1'}

In [17]:
%%time 
images = zarr_group.images[:]

CPU times: user 38 s, sys: 58.3 s, total: 1min 36s
Wall time: 5min 18s


In [18]:
v = napari.Viewer()

v.add_image(images, channel_axis=1)

[<Image layer 'Image' at 0x7f4c9a571bb0>,
 <Image layer 'Image [1]' at 0x7f4e5cbe7f40>]

In [19]:
print()

In [21]:
images.shape

(154, 2, 3, 6048, 6048)

In [82]:
# Loop through assay_layout
for (row, column), data in tqdm(assay_layout.iterrows(), total=len(assay_layout)):

    if (row, column) == (3,4):
        continue

    # First set of images without compression
    output_fn = f'/mnt/NEMO/home/users/dayn/macrohet_nemo/{expt_ID}/acquisition/zarr/{row, column}.zarr'
    os.makedirs(os.path.dirname(output_fn), exist_ok=True)    
    # if not os.path.exists(output_fn):

    images = tile.compile_mosaic(image_dir, metadata, row, column).compute()

    store = zarr.DirectoryStore(output_fn)

    # rechunk for saving 
    # images = images.rechunk((150, 2, 3, 2016, 2016)).
    images = images.rechunk((150, 2, 3, 1000, 1000))  # Adjust the chunk sizes as needed

    # Save Dask array to Zarr without compression
    images.to_zarr(store, overwrite=True, group='images', compute=True, codec=zarr.Blosc(cname='lz4', clevel=5))
    
    # images.to_zarr(store, overwrite=True, group='images', compute=True)

    zarr_group = zarr.open(store)
    zarr_group.attrs['Row'] = row
    zarr_group.attrs['Column'] = column
    for key, i in zip(data.keys(), data):
        zarr_group.attrs[key] = i

        # # Second set of images with Blosc-LZ4-Bitshuffle-8 compression
        # output_fn_compressed = f'/mnt/NEMO/home/users/dayn/macrohet_nemo/{expt_ID}/acquisition/zarr/compressed/{row, column}_compressed.zarr'
        # # make dirs
        # os.makedirs(os.path.dirname(output_fn_compressed), exist_ok=True)    
        
        # store_compressed = zarr.DirectoryStore(output_fn_compressed)
    
        # # Specify Blosc-LZ4-Bitshuffle-8 compression options
        # compressor = zarr.Blosc(cname='blosclz', clevel=5, shuffle=zarr.Blosc.BITSHUFFLE)
        
        # # Save Dask array to Zarr with Blosc-LZ4-Bitshuffle-8 compression
        # images.to_zarr(store_compressed, overwrite=True, group='images', compressor=compressor,)# compute=True,)
    
        # zarr_group_compressed = zarr.open(store_compressed)
        # zarr_group_compressed.attrs['Row'] = row
        # zarr_group_compressed.attrs['Column'] = column
        # for key, i in zip(data.keys(), data):
        #     zarr_group_compressed.attrs[key] = i

  0%|          | 0/42 [00:00<?, ?it/s]

/home/dayn/miniconda3/envs/brassica/lib/python3.9/site-packages/zarr/creation.py:295: UserWarning: ignoring keyword argument 'group'
  warn("ignoring keyword argument %r" % k)
/home/dayn/miniconda3/envs/brassica/lib/python3.9/site-packages/zarr/creation.py:295: UserWarning: ignoring keyword argument 'codec'
  warn("ignoring keyword argument %r" % k)
/home/dayn/miniconda3/envs/brassica/lib/python3.9/site-packages/zarr/creation.py:295: UserWarning: ignoring keyword argument 'group'
  warn("ignoring keyword argument %r" % k)
/home/dayn/miniconda3/envs/brassica/lib/python3.9/site-packages/zarr/creation.py:295: UserWarning: ignoring keyword argument 'codec'
  warn("ignoring keyword argument %r" % k)
/home/dayn/miniconda3/envs/brassica/lib/python3.9/site-packages/zarr/creation.py:295: UserWarning: ignoring keyword argument 'group'
  warn("ignoring keyword argument %r" % k)
/home/dayn/miniconda3/envs/brassica/lib/python3.9/site-packages/zarr/creation.py:295: UserWarning: ignoring keyword argu

ValueError: total size of new array must be unchanged